Downlaod code and dataset

In [1]:
#Download code
!git clone https://github.com/HRL-Mike/PitVQA.git

#Download Dataset
!mkdir /content/PitVQA/datasets
%cd /content/PitVQA/datasets
!gdown --id 1FoAEY_u0PTAlrscjEifi2om15A83wL78

# Unzipping the VQA EndoVis18 Dataset
!unzip -q EndoVis-18-VQA.zip
%cd /content/PitVQA

Cloning into 'PitVQA'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 78 (delta 20), reused 0 (delta 0), pack-reused 0
Receiving objects: 100% (78/78), 107.53 KiB | 7.17 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/PitVQA/datasets
/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:132: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1FoAEY_u0PTAlrscjEifi2om15A83wL78
From (redirected): https://drive.google.com/uc?id=1FoAEY_u0PTAlrscjEifi2om15A83wL78&confirm=t&uuid=b10550b9-7d23-48c5-b64b-73d8a7fa1092
To: /content/PitVQA/datasets/EndoVis-18-VQA.zip
100% 2.71G/2.71G [01:23<00:00, 32.5MB/s]
/content/PitVQA


Installing requirements

In [2]:
!pip install -q transformers
!pip install -q timm
!pip install -q fairscale

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
%cd /content/PitVQA
import os
import sys
import argparse
from tqdm import tqdm
import torch
from torch import nn
from torch import optim
from torch import optim
from torch.nn import CrossEntropyLoss
from torch.utils.data  import DataLoader
import torch.nn.functional as F
from transformers import VisualBertConfig, GPT2Config
from transformers import VisualBertModel, GPT2Model, ViTModel, SwinModel
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch.backends.cudnn as cudnn
from torchvision import models

from dataloader.Endo18_dataloader import EndoVis18VQAGPTClassification
from utils import AverageMeter, save_clf_checkpoint, adjust_learning_rate, model_url
from BLIP.models.med import BertConfig, BertModel, BertLMHeadModel
from BLIP.models.blip import create_vit, init_tokenizer, load_checkpoint
from utils import calc_acc, calc_precision_recall_fscore, calc_classwise_acc

from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from transformers import ViTModel


def seed_everything(seed=21):
    '''
    Set random seed for reproducible experiments
    Inputs: seed number
    '''
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class PitVQANet(nn.Module):
    def __init__(self,
          med_config='/content/PitVQA/BLIP/configs/med_config.json',
          num_class=18, # 18 or 59
          image_size=224,
          vit='base',
        ):
        super().__init__()

        # visual encoder
        self.visual_encoder, vision_width = create_vit(vit, image_size, use_grad_checkpointing=False,
                                  ckpt_layer=0, drop_path_rate=0.1)  # visual encoder
        # tokenizer
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token  # end of string

        # text encoder
        encoder_config = BertConfig.from_json_file(med_config)
        encoder_config.vocab_size = self.tokenizer.vocab_size  # 30524 --> 50257
        encoder_config.encoder_width = vision_width
        self.text_encoder = BertModel(config=encoder_config, add_pooling_layer=False)

        # decoder
        # change from BertLMHeadModel to GPT2Model
        self.gpt_decoder = GPT2Model.from_pretrained('gpt2')

        # intermediate_layers
        self.intermediate_layer = nn.Linear(768, 512)
        self.se_layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.Sigmoid()
        )
        self.LayerNorm = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.2)

        # classifier
        self.classifier = nn.Linear(512, num_class)

    def forward(self, image, question):
        # visual encoder
        image_embeds = self.visual_encoder(image).to(device)
        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(image.device)

        # text encoder
        encoder_question = self.tokenizer(question, return_tensors="pt", truncation=True,
                          padding='longest', max_length=25).to(image.device)

        text_output = self.text_encoder(input_ids=encoder_question.input_ids,
                          attention_mask=encoder_question.attention_mask,
                          encoder_hidden_states=image_embeds,
                          encoder_attention_mask=image_atts,
                          return_dict=True)
        text_embeds = text_output.last_hidden_state

        # text decoder
        gpt_output = self.gpt_decoder(inputs_embeds=text_embeds,
                        encoder_attention_mask=encoder_question.attention_mask)
        decoder_output = gpt_output.last_hidden_state

        # average pool
        decoder_output = decoder_output.swapaxes(1, 2)
        decoder_output = F.adaptive_avg_pool1d(decoder_output, 1)
        decoder_output = decoder_output.swapaxes(1, 2).squeeze(1)

        out = self.intermediate_layer(decoder_output)
        out = torch.mul(out, self.se_layer(out))
        out = self.LayerNorm(out)
        out = self.dropout(out)

        # classification layer
        out = self.classifier(out)

        return out


def train(train_dataloader, model, criterion, optimizer, epoch, device):
    model.train()
    total_loss = 0.0
    label_true = None
    label_pred = None
    label_score = None

    for i, (_, images, questions, labels) in enumerate(tqdm(train_dataloader),0):
        # labels
        labels = labels.to(device)
        outputs = model(image=images.to(device), question=questions)  # questions is a tuple
        loss = criterion(outputs, labels)  # calculate loss
        optimizer.zero_grad()
        loss.backward()  # calculate gradient
        optimizer.step()  # update parameters

        # print statistics
        total_loss += loss.item()

        scores, predicted = torch.max(F.softmax(outputs, dim=1).data, 1)
        if label_true is None:  # accumulate true labels of the entire training set
            label_true = labels.data.cpu()
        else:
            label_true = torch.cat((label_true, labels.data.cpu()), 0)
        if label_pred is None:  # # accumulate pred labels of the entire training set
            label_pred = predicted.data.cpu()
        else:
            label_pred = torch.cat((label_pred, predicted.data.cpu()), 0)
        if label_score is None:
            label_score = scores.data.cpu()
        else:
            label_score = torch.cat((label_score, scores.data.cpu()), 0)

    # loss and acc
    acc, c_acc = calc_acc(label_true, label_pred), calc_classwise_acc(label_true, label_pred)
    precision, recall, f_score = calc_precision_recall_fscore(label_true, label_pred)
    print(f'Train: epoch: {epoch} loss: {total_loss} | Acc: {acc} | '
        f'Precision: {precision} | Recall: {recall} | F1 Score: {f_score}')
    return acc


def validate(val_loader, model, criterion, epoch, device):
    model.eval()
    total_loss = 0.0
    label_true = None
    label_pred = None
    label_score = None
    file_names = list()

    with torch.no_grad():
        for i, (file_name, images, questions, labels) in enumerate(tqdm(val_loader),0):
            # label
            labels = labels.to(device)

            # model forward pass
            outputs = model(image=images.to(device), question=questions)

            # loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            scores, predicted = torch.max(F.softmax(outputs, dim=1).data, 1)
            label_true = labels.data.cpu() if label_true is None else torch.cat((label_true, labels.data.cpu()), 0)
            label_pred = predicted.data.cpu() if label_pred is None else torch.cat((label_pred, predicted.data.cpu()), 0)
            label_score = scores.data.cpu() if label_score is None else torch.cat((label_score, scores.data.cpu()), 0)
            for f in file_name:
                file_names.append(f)  # not used

    acc = calc_acc(label_true, label_pred)
    c_acc = 0.0
    precision, recall, f_score = calc_precision_recall_fscore(label_true, label_pred)
    print(f'Test: epoch: {epoch} test loss: {total_loss} | test acc: {acc} | '
        f'test precision: {precision} | test recall: {recall} | test F1: {f_score}')
    return acc, c_acc, precision, recall, f_score


def get_arg():
    parser = argparse.ArgumentParser(description='VisualQuestionAnswerClassification')

    # Training parameters
    parser.add_argument('--epochs', type=int, default=60, help='number of epochs to train for (if early stopping is not triggered).')  # 80, 26
    parser.add_argument('--batch_size', type=int, default=40, help='batch_size')
    parser.add_argument('--workers', type=int, default=8, help='for data-loading')

    parser.add_argument('--lr', type=float, default=0.00002, help='0.00001, 0.000005')
    parser.add_argument('--checkpoint_dir', default='checkpoints/pitvqa_2e-5', help='path for checkpoints')
    parser.add_argument('--question_len', default=25, help='25')

    parser.add_argument('--num_class', default=18, help='18/59')

    if 'ipykernel' in sys.modules:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()
    return args


def pit_vqa_custom(pretrained='', **kwargs):
    model = PitVQANet(**kwargs)
    if pretrained:
        model, msg = load_checkpoint(model, pretrained)
    return model


/content/PitVQA


In [4]:
if __name__ == '__main__':
    seed_everything()
    args = get_arg()
    args.lr = 0.00002 # 2e-5
    args.epochs = 60
    args.checkpoint_dir='checkpoints/pitvqa_2e-5'
    args.batch_size=32
    os.makedirs('checkpoints/', exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # cudnn.benchmark = True # set to true only if inputs to model are fixed size; otherwise lot of computational overhead
    print(f'device: {device}')

    # best model initialize
    start_epoch = 1
    best_epoch = [0]
    best_results = [0.0]
    epochs_since_improvement = 0

    # fold-1
    # train_seq = [2, 3, 4, 6, 7, 9, 10, 11, 12, 14, 15]
    # val_seq = [1, 5, 16]
    # fold-2
    # train_seq = [1, 2, 3, 5, 6, 7, 9, 11, 14, 15, 16]
    # val_seq = [4, 10, 12]
    # fold-3
    train_seq = [1, 3, 4, 5, 6, 7, 10, 11, 12, 14, 16]
    val_seq = [2, 9, 15]
    print(f'current train seq: {train_seq}')
    print(f'current val seq: {val_seq}')

    # data location
    folder_head = 'datasets/EndoVis-18-VQA/seq_'
    folder_tail = '/vqa/Classification/*.txt'

    # dataloader
    train_dataset = EndoVis18VQAGPTClassification(train_seq, folder_head, folder_tail)
    train_dataloader = DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=8)
    val_dataset = EndoVis18VQAGPTClassification(val_seq, folder_head, folder_tail)
    val_dataloader = DataLoader(dataset=val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=8)

    image_size = 224  # model image size
    model = pit_vqa_custom(pretrained=model_url, image_size=image_size, vit='base')
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss().to(device)

    print('Start training.')
    for epoch in range(start_epoch, args.epochs+1):
      if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
          adjust_learning_rate(optimizer, 0.8)
      # train
      train_acc = train(train_dataloader=train_dataloader, model=model, criterion=criterion,
                optimizer=optimizer, epoch=epoch, device=device)
      # validation
      test_acc, test_c_acc, test_precision, test_recall, test_f_score \
          = validate(val_loader=val_dataloader, model=model,
                criterion=criterion, epoch=epoch, device=device)
      if test_acc >= best_results[0]:
          print('Best Epoch:', epoch)
          epochs_since_improvement = 0
          best_results[0] = test_acc
          best_epoch[0] = epoch
          save_clf_checkpoint(args.checkpoint_dir, epoch, epochs_since_improvement,
                    model, optimizer, best_results[0], final_args=None)
    print('End training.')


device: cuda
current train seq: [1, 3, 4, 5, 6, 7, 10, 11, 12, 14, 16]
current val seq: [2, 9, 15]
Total files: 1587 | Total question: 9493
Total files: 420 | Total question: 2290


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Train: epoch: 5 loss: 83.38148133829236 | Acc: 0.9050879595491415 | Precision: 0.8659574625628152 | Recall: 0.7497716326104331 | F1 Score: 0.7710893366595419


100%|██████████| 72/72 [00:19<00:00,  3.71it/s]


Test: epoch: 5 test loss: 97.14791637659073 | test acc: 0.6934497816593886 | test precision: 0.6128739612967795 | test recall: 0.545460877073066 | test F1: 0.38418273912317025
Best Epoch: 5


 21%|██        | 63/297 [00:50<03:07,  1.25it/s]


KeyboardInterrupt: 